<a href="https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. Create local directories
os.makedirs('../../data/raw', exist_ok=True)
dataset_path = '../../data/raw/content_refresh_anonymized.csv'

# 2. Retrieve HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

# 3. Download the actual Parquet file from Hugging Face Hub
if not os.path.exists(dataset_path):
    print("Downloading dataset from Hugging Face Hub...")
    try:
        # Download dim_content.parquet from the repository
        downloaded_path = hf_hub_download(
            repo_id="FlyRank/internship-warehouse",
            filename="dim_content.parquet",
            repo_type="dataset",
            token=hf_token
        )
    except Exception as e:
        # Fallback to sample file if dim_content is absent
        downloaded_path = hf_hub_download(
            repo_id="FlyRank/internship-warehouse",
            filename="fact_content_daily_performance_sample.parquet",
            repo_type="dataset",
            token=hf_token
        )

    # Read Parquet and save locally to match the notebook's expected path
    df_download = pd.read_parquet(downloaded_path)
    df_download.to_csv(dataset_path, index=False)
    print(f"Download complete! Saved {len(df_download):,} rows locally.")

# 4. Connect DuckDB
con = duckdb.connect()
print("Setup complete. DuckDB connected successfully!")

Enter your Hugging Face READ token: ··········


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Download complete! Saved 519,606 rows locally.
Setup complete. DuckDB connected successfully!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Plain Words:Prioritize content pages for refresh based on content staleness—measuring the number of days elapsed between content_updated_date and our evaluation anchor date (2026-03-31).Reason Codes:STALE_CONTENT: Content updated > 180 days ago relative to evaluation anchor.FRESH_CONTENT: Content updated $\le$ 180 days ago.

In [3]:
s1_check = con.execute(f"""
    SELECT
        CASE
            WHEN DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31') > 180 THEN 'Stale (>180d)'
            ELSE 'Fresh (<=180d)'
        END AS staleness,
        COUNT(*) AS n
    FROM '{dataset_path}'
    GROUP BY 1
    ORDER BY 1
""").df()

print("Signal 1 Verification (Staleness Bucket Counts):")
print(s1_check)

Signal 1 Verification (Staleness Bucket Counts):
        staleness       n
0  Fresh (<=180d)  431753
1   Stale (>180d)   87853


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

aseline Score Formula: baseline\_score=days\_since\_last\_update365.0 Outputs the ranked priority queue directly to work/outputs/baseline_action_score.csv.

In [4]:
import os

# Create directory if missing
os.makedirs('../../work/outputs', exist_ok=True)

# Generate ranked queue using content_hash_id
df_queue = con.execute(f"""
    SELECT
        content_hash_id AS content_id,
        COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) AS days_since_last_update,
        ROUND(COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) / 365.0, 4) AS baseline_score,
        CASE
            WHEN COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) > 180 THEN 'STALE_CONTENT'
            ELSE 'FRESH_CONTENT'
        END AS reason_code,
        CASE
            WHEN COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) > 180 THEN 'REFRESH_CONTENT'
            ELSE 'MONITOR'
        END AS action_label
    FROM '{dataset_path}'
    ORDER BY baseline_score DESC
""").df()

# Export CSV to work/outputs/baseline_action_score.csv
csv_out = '../../work/outputs/baseline_action_score.csv'
df_queue.to_csv(csv_out, index=False)

print(f"Ranked queue successfully written to {csv_out} ({len(df_queue):,} rows).")
df_queue.head(5)

Ranked queue successfully written to ../../work/outputs/baseline_action_score.csv (519,606 rows).


,content_id,days_since_last_update,baseline_score,reason_code,action_label
0,content_e406c6f8fb9b5c12,519,1.4219,STALE_CONTENT,REFRESH_CONTENT
1,content_e4224cd2d80b9287,519,1.4219,STALE_CONTENT,REFRESH_CONTENT
2,content_e4b107176591a279,519,1.4219,STALE_CONTENT,REFRESH_CONTENT
3,content_e4cb7f88eff00589,519,1.4219,STALE_CONTENT,REFRESH_CONTENT
4,content_e54b36111a4ecfdd,519,1.4219,STALE_CONTENT,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df_queue.head(20)
print(top20[['content_id', 'days_since_last_update', 'baseline_score', 'reason_code', 'action_label']])

                  content_id  days_since_last_update  baseline_score  \
0   content_e406c6f8fb9b5c12                     519          1.4219   
1   content_e4224cd2d80b9287                     519          1.4219   
2   content_e4b107176591a279                     519          1.4219   
3   content_e4cb7f88eff00589                     519          1.4219   
4   content_e54b36111a4ecfdd                     519          1.4219   
5   content_e58183d2510e14e4                     519          1.4219   
6   content_e5a06707f27ab28f                     519          1.4219   
7   content_e5f256a4519fdd28                     519          1.4219   
8   content_e5f955b5c1dc3673                     519          1.4219   
9   content_e6299c63cdb26958                     519          1.4219   
10  content_e634be5d2eda8c31                     519          1.4219   
11  content_e63dad5cc4fabf57                     519          1.4219   
12  content_e651e3a84f55575a                     519          1.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Leakage Verification Check
leaked_columns = ['trend_direction', 'trend_pct', 'is_declining_label']
score_inputs = ['days_since_last_update']

assert not any(col in score_inputs for col in leaked_columns), "Leakage detected!"
print("Leakage check passed: Scoring relies exclusively on honest pre-decision features.")

# 2. Inspect Weak Picks (Oldest pages with zero/low priority signals)
weak_picks_check = df_queue.sort_values(by='baseline_score', ascending=False).head(10)
print("\n--- Sample Scored Picks for Verification ---")
print(weak_picks_check[['content_id', 'days_since_last_update', 'baseline_score', 'reason_code']])

Leakage check passed: Scoring relies exclusively on honest pre-decision features.

--- Sample Scored Picks for Verification ---
                  content_id  days_since_last_update  baseline_score  \
12  content_e651e3a84f55575a                     519          1.4219   
35  content_e9bf110ceaf94d18                     519          1.4219   
31  content_e9055130ff7fa814                     519          1.4219   
0   content_e406c6f8fb9b5c12                     519          1.4219   
30  content_e8b2fe10432b4b22                     519          1.4219   
50  content_eda7ed7457ac9418                     519          1.4219   
49  content_ecf267e24a379986                     519          1.4219   
16  content_e6e5b99d0a0cf21d                     519          1.4219   
47  content_ecab3e9b269555fa                     519          1.4219   
46  content_eca0c07037b4e194                     519          1.4219   

      reason_code  
12  STALE_CONTENT  
35  STALE_CONTENT  
31  STALE_CONTENT  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.